# Lab 7 — Training a YOLO26 object detector

**Module:** 7144COMP — Deep Learning Concepts and Techniques  
**Week:** 7  
**Estimated time:** 240 minutes

---

## ⚠ Hardware requirement

**This lab requires a CUDA-capable NVIDIA GPU.** The container is built from a CUDA base image and expects to be run with GPU passthrough enabled (`deploy.resources.devices: nvidia` in `docker-compose.yml`). Tested configurations: RTX 3090 / 4090 / 5090.

If you do not have a GPU, you can still complete the conceptual sections and read through the code, but the training cells will fail at the device-check step.

## Learning outcomes

By the end of this lab you should be able to:

1. Verify GPU availability and select the right device for training.
2. Explain what makes object detection a fundamentally different problem from image classification, and where YOLO sits in the family of detection architectures.
3. Train a YOLO26x detector on a custom YOLO-format dataset using the Ultralytics Python API.
4. Interpret object detection metrics — precision, recall, **mAP50**, **mAP50-95** — and explain what each tells you about model behaviour.
5. Run inference on new images and visualise the resulting bounding boxes with class labels and confidence scores.
6. Export the trained model to ONNX for deployment in non-PyTorch environments.

## Prerequisites

- **Lab 6** completed — your annotated dataset and `data.yaml` are what we'll train on. *If you didn't complete Lab 6 we provide a fallback in Section 2.*
- **Labs 1–5** completed — you should be comfortable with PyTorch, training loops, and evaluation metrics.
- The textbook *Applied Deep Learning* (Fergus & Chalmers), Chapter 7 — object detection.
- Lecture 7: *Single-shot detectors, anchor-free heads, and the post-NMS era of YOLO*.

## The thread from last week

In Lab 6 you produced a clean YOLO-format dataset: images on disk, bounding-box annotations in `.txt` files, a `data.yaml` that ties it all together. **This week we use that dataset to train a real, production-grade object detector.** Everything we've built across six weeks — tensor mechanics, training loops, augmentation, CNN architecture — converges here.

## Useful references

- [Ultralytics YOLO26 documentation](https://docs.ultralytics.com/models/yolo26)
- [Ultralytics performance metrics guide](https://docs.ultralytics.com/guides/yolo-performance-metrics/)
- The original *You Only Look Once* paper: Redmon et al. (2016)
- For end-to-end NMS-free detection: Wang et al. (2024), *YOLOv10: Real-Time End-to-End Object Detection*

**License note:** Ultralytics ships YOLO26 under **AGPL-3.0**. This is fine for academic use and for personal projects, but is *not* permissive — it has copyleft obligations if you deploy a network service. Commercial users may need an Ultralytics Enterprise license. We discuss the deployment implications in the reflection questions.

---

## 1. GPU sanity check

Before we go anywhere, verify the GPU is visible from inside the container. If this fails, you have a host-side problem (driver, NVIDIA Container Toolkit, or `docker-compose.yml` GPU block) — fix it before continuing.

On a 3090/4090/5090 the cell below should print something like `NVIDIA GeForce RTX 4090` with 24 GB of memory.

In [ ]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA device found. Check that:\n"
        "  1. Your host has a working NVIDIA driver (try `nvidia-smi` outside the container).\n"
        "  2. The NVIDIA Container Toolkit is installed (Docker Desktop auto-installs on Windows).\n"
        "  3. docker-compose.yml has the `deploy.resources.devices: nvidia` block enabled.\n"
        "  4. You started the container with `docker compose up` (NOT `docker run` without --gpus)."
    )

print(f"Device count:    {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name}")
    print(f"          {props.total_memory / 1e9:.1f} GB memory")
    print(f"          compute capability {props.major}.{props.minor}")

## 2. The dataset

**Best case:** you completed Lab 6 and have a `data.yaml` plus a populated `images/train/`, `images/val/`, `images/test/`, and corresponding `labels/` folders in `labs/lab06_image_annotation/data/`. Set `DATA_YAML` to point at it.

**Fallback:** if you didn't complete Lab 6 (or only labelled a few images), we ship a pre-annotated copy of the 15 starter images as a fallback. The fallback dataset is tiny and only useful for *checking the training pipeline works* — to train a model that's actually useful for detecting real UK wildlife, you need a real dataset of at least a few hundred images per class.

Set `USE_FALLBACK = True` to use the bundled dataset; `False` to use your Lab 6 work.

In [ ]:
from pathlib import Path
import shutil

USE_FALLBACK = True   # set to False once you've completed Lab 6 properly

if USE_FALLBACK:
    # Build (or rebuild) the fallback dataset in case it's not already on disk.
    fallback_root = Path("data/fallback_dataset")
    if not (fallback_root / "data.yaml").is_file():
        import subprocess
        print("Building fallback dataset...")
        result = subprocess.run(
            ["python", "data/build_fallback_dataset.py"],
            capture_output=True, text=True,
        )
        print(result.stdout[-500:])
        if result.returncode != 0:
            raise RuntimeError(f"Fallback build failed:\n{result.stderr}")
    DATA_YAML = (fallback_root / "data.yaml").resolve()
else:
    # Use your Lab 6 dataset.
    DATA_YAML = (Path("..") / "lab06_image_annotation" / "data" / "data.yaml").resolve()

assert DATA_YAML.is_file(), f"data.yaml not found at {DATA_YAML}"
print(f"Using dataset: {DATA_YAML}")
print("\nContents:")
print(DATA_YAML.read_text())

### A look at the dataset before training

Always inspect your data before training. Below we render a few annotated samples — exactly the same pattern you used in Lab 6's verification step. Bad annotations here mean a bad model; if anything looks wrong, fix it before spending GPU time.

In [ ]:
import yaml
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

with open(DATA_YAML) as f:
    cfg = yaml.safe_load(f)

classes = cfg["names"]
if isinstance(classes, dict):
    # Ultralytics accepts both list and dict forms — normalise.
    classes = [classes[i] for i in sorted(classes.keys())]
print(f"Classes: {classes}")

CLASS_COLOURS = {
    0: "#d6336c", 1: "#f59f00", 2: "#2b8a3e", 3: "#9a59f4", 4: "#1c7ed6",
}

dataset_root = Path(cfg["path"])
train_imgs = sorted((dataset_root / cfg["train"]).glob("*.jpg"))
train_lbls = dataset_root / cfg["train"].replace("images", "labels")

sample = random.Random(7144).sample(train_imgs, k=min(6, len(train_imgs)))

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, img_path in zip(axes.flat, sample):
    img = Image.open(img_path)
    W, H = img.size
    ax.imshow(img); ax.axis("off"); ax.set_title(img_path.name, fontsize=10)
    lbl_path = train_lbls / f"{img_path.stem}.txt"
    if lbl_path.is_file():
        for line in lbl_path.read_text().splitlines():
            if not line.strip(): continue
            cls, cx, cy, w, h = line.split()
            cls = int(cls); cx, cy, w, h = map(float, (cx, cy, w, h))
            x = (cx - w/2) * W; y = (cy - h/2) * H
            colour = CLASS_COLOURS.get(cls, "#888")
            ax.add_patch(patches.Rectangle((x, y), w*W, h*H,
                                            linewidth=2, edgecolor=colour, facecolor="none"))
            ax.text(x+3, y-4, classes[cls], color="white", fontsize=10, fontweight="bold",
                    bbox=dict(boxstyle="square,pad=0.2", facecolor=colour, edgecolor="none"))
plt.tight_layout()
plt.show()

## 3. What is YOLO actually doing?

Before we run `model.train()` and get a blizzard of metrics, let's anchor the conceptual move from classification to detection.

<img src="assets/yolo_concept.svg" alt="Classification vs detection" width="880"/>

**Classification** (Labs 3–5) takes one image and produces *one* label — a single softmax over class probabilities. There's no concept of *where* anything is.

**Detection** (Lab 7) takes one image and produces *a variable number* of bounding boxes, each with its own class id and confidence score. The model must learn to:

1. **Localise** — predict where every object is, as (cx, cy, w, h).
2. **Classify** — predict what each object is.
3. **Count** — implicitly, decide how many objects exist (zero is also a valid answer).

YOLO's contribution, in 2016, was to do all of this in a **single forward pass** of one neural network — hence "You Only Look Once". Before YOLO, detection used two-stage pipelines (R-CNN, Faster R-CNN) that proposed regions first, then classified them. YOLO collapsed it into one.

**YOLO26's headline improvements** over earlier versions:

- **End-to-end NMS-free inference.** Older YOLO models produced many overlapping predictions per object, then ran non-maximum suppression (NMS) as a post-processing step to pick the best ones. YOLO26 (following YOLOv10) trains the network itself to produce de-duplicated predictions directly. Faster inference, simpler deployment.
- **MuSGD optimizer.** A hybrid of SGD and the Muon optimizer (originally developed for large language model training). More stable convergence than Adam or pure SGD on detection tasks.
- **ProgLoss + STAL.** Specific loss functions designed to improve small-object detection performance.

## 4. Object detection metrics

Detection has its own evaluation vocabulary, distinct from classification. You met IoU in Lab 6 — it's the foundation.

**IoU (Intersection over Union):** for any predicted box and any ground-truth box, IoU is the overlap area divided by the union area. Range 0 (no overlap) to 1 (perfect overlap). A prediction is counted as a 'true positive' if its IoU with a ground-truth box exceeds some threshold (typically 0.5).

**Precision & recall, per class:**
- *Precision* = of the boxes the model predicts, what fraction match a real object?
- *Recall* = of the real objects, what fraction does the model find?

**mAP50 (mean Average Precision at IoU 0.50):** the standard 'one number for detection quality'. It varies the model's confidence threshold from 0 to 1, plots precision vs recall, takes the area under the curve, averages across classes. mAP50 = 0.9 means the model is very good at finding objects with reasonable localisation.

**mAP50-95 (mean Average Precision averaged across IoU thresholds 0.50 to 0.95):** a *much* stricter metric. It computes AP at IoU = 0.50, 0.55, 0.60, ..., 0.95, and averages all ten. mAP50-95 = 0.9 means the model not only finds the objects but localises them *very precisely*. This is the headline metric in the COCO benchmark and the one Ultralytics reports as the main score.

**Why mAP50-95 matters.** A model that can find a fox but draws sloppy boxes (50-60% IoU) will score well on mAP50 but poorly on mAP50-95. For applications where the *box* matters (counting, measuring, cropping), the latter is what you care about. For 'is there a fox in this image at all' applications, mAP50 is enough.

## 5. Loading a pretrained YOLO26x model

We start from a **pretrained checkpoint** — `yolo26x.pt`, which was trained on the COCO dataset (80 everyday object classes including 'dog', 'horse', 'bird', 'sheep'). We then *fine-tune* it on our 5-class UK wildlife dataset. This is the canonical transfer-learning move: the lower layers have already learned generic visual features that transfer to almost any natural-image task, and we only need to retrain the head and adjust the rest slightly.

**Why YOLO26x and not a smaller variant?** Because you specified the requirement. The size variants are:

| Variant      | Params | Best for |
|--------------|--------|----------|
| `yolo26n.pt` | ~3M    | Edge devices, real-time on CPU |
| `yolo26s.pt` | ~10M   | Mobile, faster training |
| `yolo26m.pt` | ~25M   | Balanced |
| `yolo26l.pt` | ~45M   | Server-side, higher accuracy |
| `yolo26x.pt` | ~70M   | Maximum accuracy, GPU required |

For a tiny dataset like ours, `yolo26x` is *overkill* — `yolo26n` would probably train just as well on 10 images. But we're here to learn the workflow at full scale, so x it is. Exercise 2 explores smaller variants.

In [ ]:
from ultralytics import YOLO

# First time you run this, Ultralytics downloads yolo26x.pt (~280 MB) to the
# current working directory. Subsequent runs use the cached file.
model = YOLO("yolo26x.pt")

print(f"Model type: {type(model).__name__}")
print(f"Task:       {model.task}")
n_params = sum(p.numel() for p in model.model.parameters())
print(f"Parameters: {n_params:,}")

## 6. Training

Now the headline event. The Ultralytics `model.train()` call hides a lot of machinery — augmentation, batching, optimizer scheduling, validation, checkpointing — behind one method. We will walk through every parameter.

> ⏱ **Time estimate.** On a 3090/4090/5090, 50 epochs on the fallback dataset of 15 images takes 3–5 minutes. On a larger custom dataset of 500 images, expect 30–60 minutes. The first epoch is always slower than the rest because Ultralytics does initial setup and dataset caching.

**Parameters explained:**

- `data` — path to your `data.yaml`.
- `epochs=50` — how many full passes over the training set.
- `imgsz=640` — resize all input images to 640×640 for training. YOLO is fully convolutional so it would handle other sizes, but 640 is the canonical size for the COCO pretrained weights.
- `batch=16` — batch size. For yolo26x at 640×640, this uses about 14-18 GB of VRAM. **Reduce to 8 if you OOM**, increase to 32 on a 32GB GPU like the 5090.
- `device=0` — use GPU 0. Use `[0, 1]` for multi-GPU.
- `project` and `name` — where Ultralytics writes outputs. We pin this to a known location.
- `patience=20` — Ultralytics' built-in early stopping. Stops if validation metric stops improving for 20 epochs.
- `seed=7144` — module-wide reproducibility convention.
- `verbose=True` — show the per-epoch metrics.

In [ ]:
import time

t0 = time.perf_counter()
results = model.train(
    data=str(DATA_YAML),
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    project="runs",
    name="uk_wildlife",
    exist_ok=True,    # overwrite if previous run is in that folder
    patience=20,
    seed=7144,
    verbose=True,
    pretrained=True,
)
elapsed = time.perf_counter() - t0
print(f"\nTraining done in {elapsed/60:.1f} minutes.")
print(f"Results saved to: {results.save_dir}")

## 7. What did Ultralytics write to disk?

Ultralytics produces a lot of output. Knowing what's where is the difference between feeling overwhelmed and feeling productive.

In [ ]:
run_dir = Path(results.save_dir)
print(f"Training run directory: {run_dir}\n")
for p in sorted(run_dir.iterdir()):
    if p.is_file():
        size = p.stat().st_size
        print(f"  {p.name:<35s} {size/1024:.0f} KB")
    elif p.is_dir():
        n = sum(1 for _ in p.iterdir())
        print(f"  {p.name}/ ({n} files)")

Key files:

- **`weights/best.pt`** — the model checkpoint with the best validation mAP across all epochs. This is the model you deploy.
- **`weights/last.pt`** — the most-recent checkpoint, useful for resuming a stopped run.
- **`results.png`** / **`results.csv`** — training curves (loss, precision, recall, mAP per epoch).
- **`confusion_matrix.png`** — class-vs-class confusion at the chosen confidence threshold.
- **`PR_curve.png`** — precision-recall curve, one line per class, averaged for mAP.
- **`labels.jpg`** / **`labels_correlogram.jpg`** — dataset diagnostics (class distribution, bounding-box size distribution).
- **`train_batchN.jpg`** / **`val_batchN_pred.jpg`** — augmented training batches and prediction batches for visual inspection.

Let's look at the training curves first.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 9))
img = Image.open(run_dir / "results.png")
ax.imshow(img); ax.axis("off")
ax.set_title("Training curves — loss components, precision, recall, mAP50, mAP50-95")
plt.tight_layout(); plt.show()

**What to look for in the training curves:**

- *Box loss* and *cls loss* (top row) should both fall steadily — these are the two components of the detection loss.
- *Precision* should rise toward 1.0. *Recall* should also rise — these are class-aggregated.
- *mAP50* and *mAP50-95* (bottom row) tell the headline story. They should rise and then plateau.
- **If validation metrics plateau much lower than training metrics**, you're overfitting. The fallback dataset is small enough that this *will* happen — small datasets are a teaching limitation, not a model failure.

## 8. Validation on the held-out test set

Ultralytics ran validation *during* training (on the `val` split, against the best weights). Now we explicitly run it again on the `test` split — the data the model has never seen and we haven't used for early stopping. This is the honest evaluation.

In [ ]:
# Load the best checkpoint.
best_model = YOLO(str(run_dir / "weights" / "best.pt"))

# Evaluate on the test split (the `split` argument selects which entry in data.yaml to use).
metrics = best_model.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=640,
    device=0,
    project="runs",
    name="uk_wildlife_test",
    exist_ok=True,
    verbose=False,
)

print("\nHeadline metrics on test split:")
print(f"  Precision (mean): {metrics.box.mp:.4f}")
print(f"  Recall (mean):    {metrics.box.mr:.4f}")
print(f"  mAP50:            {metrics.box.map50:.4f}")
print(f"  mAP50-95:         {metrics.box.map:.4f}")

print("\nPer-class mAP50-95:")
for i, class_name in enumerate(classes):
    if i < len(metrics.box.maps):
        print(f"  {class_name:<10s} {metrics.box.maps[i]:.4f}")

### Confusion matrix

Ultralytics also writes a class-confusion plot. For detection problems this is *slightly* different from a classification confusion matrix — it includes a 'background' row/column for cases where the model predicted a box where there's nothing (false positive) or missed a box that was there (false negative).

In [ ]:
# Use Ultralytics' authoritative output dir rather than guessing the path.
# Recent versions sometimes write 'confusion_matrix_normalized.png' instead of
# (or alongside) 'confusion_matrix.png', so we glob for any match.
test_run_dir = Path(metrics.save_dir)
cm_paths = sorted(test_run_dir.glob("confusion_matrix*.png"))

if cm_paths:
    fig, axes = plt.subplots(1, len(cm_paths), figsize=(8 * len(cm_paths), 7))
    if len(cm_paths) == 1:
        axes = [axes]
    for ax, p in zip(axes, cm_paths):
        ax.imshow(Image.open(p)); ax.axis("off")
        ax.set_title(p.stem.replace("_", " ").title(), fontsize=11)
    plt.tight_layout(); plt.show()
else:
    print(f"No confusion matrix files found in {test_run_dir}. Other files there:")
    for f in sorted(test_run_dir.iterdir()):
        print(f"  {f.name}")

## 9. Inference on new images

The real test is: can the model find animals in images it has never seen? We bundle three held-out test images in `test_images/` for this purpose. You can also drop your own JPEGs into that folder.

`model.predict()` accepts a single image path, a directory, a list of paths, or even a URL or video file.

In [ ]:
test_image_dir = Path("test_images")
test_paths = sorted(test_image_dir.glob("*.jpg"))
print(f"Will predict on {len(test_paths)} images:")
for p in test_paths: print(f"  {p.name}")

predictions = best_model.predict(
    source=[str(p) for p in test_paths],
    imgsz=640,
    conf=0.25,       # ignore predictions below 25% confidence
    device=0,
    save=True,       # write annotated images to disk
    project="runs",
    name="uk_wildlife_predict",
    exist_ok=True,
    verbose=False,
)

print(f"\nAnnotated images written to: runs/uk_wildlife_predict/")

In [ ]:
import numpy as np
# Use Ultralytics' authoritative save_dir from the predict results, not a guessed path.
predict_dir = Path(predictions[0].save_dir) if predictions else None
print(f"Predict dir: {predict_dir}")

annotated = []
if predict_dir and predict_dir.is_dir():
    # Catch both jpg and png — Ultralytics' save format varies by source.
    annotated = sorted(list(predict_dir.glob("*.jpg")) + list(predict_dir.glob("*.png")))

if not annotated:
    print("No annotated images found. Diagnostic listing:")
    if predict_dir and predict_dir.is_dir():
        for f in sorted(predict_dir.iterdir()):
            print(f"  {f.name}")
else:
    # 2-per-row grid so each image is large enough to read.
    N_COLS = 2
    n_rows = (len(annotated) + N_COLS - 1) // N_COLS
    fig, axes = plt.subplots(n_rows, N_COLS, figsize=(9 * N_COLS, 7 * n_rows))
    axes = np.atleast_1d(axes).flatten()
    for ax, p in zip(axes, annotated):
        ax.imshow(Image.open(p)); ax.axis("off"); ax.set_title(p.name, fontsize=11)
    for ax in axes[len(annotated):]:
        ax.set_visible(False)
    plt.tight_layout(); plt.show()

In [ ]:
# Inspect the predictions structurally — useful for downstream work.
for r in predictions:
    print(f"\n{Path(r.path).name}")
    if r.boxes is None or len(r.boxes) == 0:
        print("  (no detections)")
        continue
    boxes = r.boxes.xyxyn.cpu().numpy()       # normalised xyxy
    confs = r.boxes.conf.cpu().numpy()
    cls_ids = r.boxes.cls.cpu().numpy().astype(int)
    for i in range(len(boxes)):
        x1, y1, x2, y2 = boxes[i]
        print(f"  {classes[cls_ids[i]]:<10s} conf={confs[i]:.3f}  "
              f"xyxy=({x1:.3f}, {y1:.3f}, {x2:.3f}, {y2:.3f})")

## 10. Export for deployment

When you want to run the trained model in production — on a server, on an edge device, in a non-Python language — you typically export it to a deployment-friendly format. **ONNX** (Open Neural Network Exchange) is the most portable: every major inference runtime supports it.

Ultralytics makes export a one-liner.

In [ ]:
onnx_path = best_model.export(format="onnx", imgsz=640, simplify=True)
print(f"Exported model to: {onnx_path}")
print(f"Size: {Path(onnx_path).stat().st_size / 1e6:.1f} MB")

**What you can do with the ONNX file:**

- Load it in **onnxruntime** for CPU or GPU inference without PyTorch.
- Convert it to **TensorRT** for maximum speed on NVIDIA hardware (`yolo export format=engine`).
- Convert it to **CoreML** for iOS deployment, **TFLite** for Android, or **OpenVINO** for Intel hardware.
- Inspect the graph in [Netron](https://netron.app/) to understand the architecture.

Verification that the ONNX file actually works follows the same pattern you'd use for any ONNX model:

In [ ]:
import onnxruntime as ort
import numpy as np

session = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
input_meta = session.get_inputs()[0]
output_meta = session.get_outputs()[0]
print(f"ONNX input:  {input_meta.name}  shape {input_meta.shape}  dtype {input_meta.type}")
print(f"ONNX output: {output_meta.name} shape {output_meta.shape}")

# Quick smoke test: a single zero input shouldn't crash.
dummy = np.zeros((1, 3, 640, 640), dtype=np.float32)
out = session.run(None, {input_meta.name: dummy})
print(f"ONNX dummy inference: output shape {out[0].shape}")

---

## 11. Exercise 1 — explore hyperparameters

Pick **one** hyperparameter and study its effect. Train two additional runs (one above the baseline, one below), record the test mAP50-95 for each, and write 3–5 sentences interpreting the result.

**Choose one of:**

- **`imgsz`** — try `imgsz=320` and `imgsz=1280`. Bigger images = more detail captured but slower. How much does image size affect performance on this dataset?
- **`epochs`** — try `epochs=20` and `epochs=100`. Where does the model start overfitting (val mAP rises then falls)?
- **`batch`** — try `batch=4` and `batch=32`. Does batch size affect *final* accuracy, or just *training stability and wall-clock*?
- **`lr0`** — try `lr0=0.001` (10× lower) and `lr0=0.05` (5× higher) than Ultralytics' default. Does either extreme break training?

*Each extra training run takes a few minutes on a GPU. Plan accordingly.*

In [ ]:
# Your code for Exercise 1 here.


*Which hyperparameter did you pick? What did you find?*




## 12. Exercise 2 — model size matters

We trained `yolo26x` (~70M parameters) on a 15-image dataset. That's *enormously* over-parameterised for the task. Modern wisdom says: **always start with the smallest variant** and only go larger if you can show the small one is the bottleneck.

Re-train using **`yolo26n.pt`** (~3M parameters, 20× smaller) with the **same hyperparameters** and the **same data**. Report:

**(a)** Test mAP50-95 — does the smaller model actually do worse on our tiny dataset, or roughly the same?

**(b)** Wall-clock training time. 

**(c)** Inference time per image (run `predict()` on the same test image with both models, measure with `time.perf_counter()`).

**(d)** Write a paragraph (4–6 sentences) on when you would and wouldn't choose the larger variant in practice. Consider: dataset size, deployment target, latency requirements, compute budget.

In [ ]:
# Your code for Exercise 2 here.


*Your written analysis:*



## 13. Exercise 3 — open-ended (pick one)

**Pick *one* and complete it fully.**

### Option A — Confidence threshold sweep

Just as in Lab 2's medical-screening exercise, the decision threshold matters. Re-run `predict()` on the test images at confidence thresholds 0.10, 0.25, 0.50, 0.75. For each, count:

- True positives (predictions that match a ground-truth box with IoU > 0.5)
- False positives (predictions with no matching ground truth)
- False negatives (ground-truth boxes with no matching prediction)

Plot precision vs recall across the four thresholds. Which threshold would you use if you were building a wildlife monitoring system where missing a fox is much worse than over-reporting one?

### Option B — Predict on a video

YOLO can process video. Either download a short clip of UK wildlife (Wikimedia Commons has many CC-licensed options) or use any phone video you have. Save it as `test_images/clip.mp4`. Then:

```python
results = best_model.predict(source="test_images/clip.mp4", save=True, conf=0.25)
```

Ultralytics will write an annotated video to `runs/uk_wildlife_predict/clip.avi`. Watch it. Comment on:

1. Frame-to-frame stability — does the same object get the same class id across consecutive frames?
2. The 'flicker' problem — boxes that appear and disappear between frames.
3. Whether you'd want object *tracking* (persistent ids across frames) on top of detection.

### Option C — Build your own real dataset

Source **50+ real UK wildlife photos** from iNaturalist, Wikimedia Commons, or your own collection. Annotate them with the Lab 6 annotator. Set `USE_FALLBACK = False`. Re-run the entire training pipeline above. Compare the test mAP against the synthetic fallback. Reflect: how much did real data improve things? What does this say about *the bottleneck* in training useful detection models?

In [ ]:
# Your code for Exercise 3 (Option A, B, or C) here.


*Which option did you pick?*


*Your written reflection:*



---

## 14. Reflection questions

**Q1.** Explain in your own words what 'pretrained on COCO' means. Why is it usually a better starting point for a new detection task than training from random weights?

**Q2.** YOLO26 is *end-to-end NMS-free*. Older YOLO models needed a post-processing step called non-maximum suppression. Briefly: what problem does NMS solve, and what is the practical benefit of training a network that no longer needs it?

**Q3.** The lab repeatedly warned that 15 images is too few for a real model. *Why* is small data such a problem for object detection specifically? (Hint: think about how many parameters yolo26x has and how many examples each parameter sees per epoch.)

**Q4.** mAP50-95 was 0.0 or close to it for some classes in your run — this is normal for tiny datasets. State at least **two** specific things you would change *first* if you wanted that number to climb to a useful production level. Be concrete.

**Q5.** Ultralytics ships YOLO26 under the **AGPL-3.0** license. Suppose you wanted to deploy your trained model as the inference engine behind a paid wildlife-monitoring SaaS product. What's the legal/practical concern, and what are your options? (You're not a lawyer; a 3-sentence answer is fine.)

*Your answers:*

**A1.** 

**A2.** 

**A3.** 

**A4.** 

**A5.** 

---

## What's next

You've now trained a real, production-grade object detector end-to-end. The pipeline you used is the same one that ships wildlife cameras, traffic monitoring systems, factory floor inspection, and a hundred other applications.

**Lab 8** will focus on the question 'now that I've trained a model, how do I deploy it?' — covering inference optimisation (TensorRT, INT8 quantisation), serving via a simple HTTP API, and monitoring real-world model performance once it's in production. We'll close the loop from research code to running system.

Before leaving today, make sure:

- [ ] Training completed successfully on the GPU
- [ ] You have completed Exercises 1, 2, and 3
- [ ] You have answered the reflection questions
- [ ] Your notebook runs **top to bottom without errors** (*Kernel → Restart and Run All*)
- [ ] `runs/uk_wildlife/weights/best.pt` exists and is the model you'd deploy
- [ ] An ONNX export was produced successfully